**Step 1: Entity and relationship extraction with LLM**

In [ ]:
import pandas as pd
import os
import time
import math

file_path = r"insert file path"
output_file_path = r"insert output file path"
save_percent = 5 

# Load existing saved file if it already exists, otherwise start from original input
if os.path.exists(output_file_path):
    df = pd.read_excel(output_file_path)
else:
    df = pd.read_excel(file_path)

# import openai
from openai import AzureOpenAI

client = AzureOpenAI(
    api_key="YOUR_API_KEY",
    api_version="YOUR_API_VERSION", # or your Azure version
    azure_endpoint="https://YOUR-AZURE-OPENAI-RESOURCE.openai.azure.com/",
)


def ask_questions(abstract, questions, system_prompts):
    responses = []
    for question, system_prompt in zip(questions, system_prompts):
        prompt_text = question + " " + str(abstract)

        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": prompt_text}
        ]
        #try block is error handling function when sth goes wrong. It tries to send request to azure
        #temperature controls randomness. temp=0 is deterministic. temp=1 is creative.
        try:
            response = client.chat.completions.create(
                model="gpt-4.1",  
                messages=messages,
                temperature=0
            )
            answer = response.choices[0].message.content
            responses.append(answer.strip())
        except Exception as e:
            print(f"Error getting response: {e}")
            responses.append("")
    return responses

# ---------------------------------------------------
# Example usage reading from Excel and saving results
# ---------------------------------------------------

questions = [" "]
system_prompts = [
    "You are a scientific literature analyzer specialized in algae, anaerobic digestion, and bioresource recovery systems. Your task is to extract all context-based or causal relationships between key entities mentioned in a research paper abstract. A relationship exists when one entity influences, produces, utilizes, enables, improves, captures, converts, or is functionally linked with another entity within the described system. Relationships may include causal relationships (A affects or leads to B), functional relationships (A is used for B), or system relationships (A operates within B or together with B). You must only consider the following entities: biogas upgrading, biogas, co2 capture, algae, anaerobic digestion, ad, co-digestion, algal biomass, photobioreactor, bioremediation, wastewater, biofuel, methane, acetate, bioplastic, pha, phb, biomaterial, HRAP, high rate algal pond, BLSS, iss, microgravity, uv, radiation, nutrient mining, food source, fertilizer production, BES, MES, MEC, immobilization, biofilm, hydrogel, vfa, electrosynthesis, biochar, electrode, biocathode, mfc, cyanobacteria, nanoparticle, pretreatment, life support, space, mars, astrobiology, survival, cell wall, biomass, co-culture, consortia, bioelectrochemical, machine learning. Carefully read the abstract, identify which of the listed entities appear or are implied, and determine all relationships between those entities. Include both direct and indirect relationships when the abstract context logically connects them. Do not invent entities not in the list. If an entity appears multiple times still include its relationships. Consider biological, engineering, environmental, and space biotechnology contexts. Include relationships even if the interaction is implied rather than explicitly stated. Output only relationship pairs using the format (entity A, entity B), (entity C, entity D), (entity E, entity F) with no explanations or additional text. If no listed entities appear in the abstract output: None."
]

if 'Extracted entities' not in df.columns:
    df['Extracted entities'] = ""

total_rows = len(df)
save_interval = math.ceil(total_rows * save_percent / 100)

try:
    for i, row in df.iterrows():
        if pd.notna(df.at[i, 'Extracted entities']) and str(df.at[i, 'Extracted entities']).strip() != "":
            continue

        os.system('cls' if os.name == 'nt' else 'clear')
        response = ask_questions(row['Abstract'], [questions[0]], [system_prompts[0]])[0]

        df.at[i, 'Extracted entities'] = response

        print(f"Response for Row {i+1}:")
        print(f"Answer to Question 2: {response}")
        progress = ((i + 1) / total_rows) * 100
        print(f"Progress: {progress:.2f}% completed")

        # autosave every 5%
        if (i + 1) % save_interval == 0:
            df.to_excel(output_file_path, index=False)
            print(f"AUTOSAVED at {progress:.2f}%")

    # Final save
    df.to_excel(output_file_path, index=False)
    print("Final save completed :)")

# saving if the file crashes
except Exception as e:
    print(f"Error: {e}")
    df.to_excel(output_file_path, index=False)
    print("Progress saved before stopping.")
    raise

**Step 3: Combine entities with similar meanings**

In [ ]:
import sys
import pandas as pd
import re
import numpy as np
import concurrent.futures
import openai
import os
import math

from openai import AzureOpenAI

# =========================
# FILE PATHS
# =========================
input_file_path = r"INPUT_FILE_PATH"
modified_file_path = r"OUTPUT_FILE_PATH"
save_percent = 5

# =========================
# LOAD EXISTING MODIFIED FILE IF PRESENT
# =========================
if os.path.exists(modified_file_path):
    df = pd.read_excel(modified_file_path, engine="openpyxl")
    print(f"Resuming from existing file: {modified_file_path}")
else:
    df = pd.read_excel(input_file_path, engine="openpyxl")
    print(f"Starting from input file: {input_file_path}")

# Make sure required columns exist
df["Extracted entities"] = df["Extracted entities"].fillna("")
if "Replacement done" not in df.columns:
    df["Replacement done"] = ""

column_values = df["Extracted entities"].astype(str).tolist()

# =========================
# AZURE CLIENT
# =========================

client = AzureOpenAI(
    api_key="YOUR_API_KEY",
    api_version="YOUR_API_VERSION",
    azure_endpoint="https://YOUR-AZURE-OPENAI-RESOURCE.openai.azure.com/",
    azure_deployment="YOUR_DEPLOYMENT_NAME"
)

##################################################
# 1) EXTRACT ENTITIES
##################################################

pattern = r"\(([^,]+), ([^)]+)\)"
entities = []
for value in column_values:
    matches = re.findall(pattern, value)
    for (e1, e2) in matches:
        entities.append(e1)
        entities.append(e2)

# Remove duplicates while preserving order
entities = list(dict.fromkeys(entities))

##################################################
# 2) EMBEDDINGS IMPLEMENTATION
##################################################

#converts text to vector embedding
def get_openai_embedding(text, model="text-embedding-3-small"):
    """Get embedding with proper error handling and API parameters"""
    try:
        response = client.embeddings.create(
            input=text,
            model=model,
            encoding_format="float"  # Explicitly request float format
        )
        return response.data[0].embedding
    except openai.APIError as e:
        print(f"API Error: {e.status_code} - {e.message}")
    except Exception as e:
        print(f"General error embedding '{text[:20]}...': {str(e)}")
    return None

# --- PARALLEL EMBEDDING REQUESTS WITH RATE LIMIT HANDLING ---
model_name = "text-embedding-3-small"  # Can change to "text-embedding-ada-002" for older model
all_embeddings = []

# Reduced max_workers to comply with OpenAI rate limits
with concurrent.futures.ThreadPoolExecutor(max_workers=10) as executor:
    future_to_entity = {
        executor.submit(get_openai_embedding, ent, model_name): ent 
        for ent in entities
    }

    for future in concurrent.futures.as_completed(future_to_entity):
        ent = future_to_entity[future]
        try:
            emb = future.result()
            all_embeddings.append((ent, emb))
        except Exception as e:
            print(f"Error processing '{ent}': {e}")
            all_embeddings.append((ent, None))

# Create embedding dictionary maintaining original entity order
emb_dict = dict(all_embeddings)
vectors = []
for ent in entities:
    emb_vec = emb_dict.get(ent)
    if emb_vec is not None:
        vectors.append(np.array(emb_vec, dtype=np.float32))
    else:
        vectors.append(None)

##################################################
# 3) COSINE SIMILARITY (VECTORIZED IN NUMPY)
##################################################
# Ensure a consistent dimensionality by replacing missing embeddings with zeros
valid_vectors = [v for v in vectors if v is not None]
if not valid_vectors:
    print("No valid embeddings found, cannot proceed.")
    sys.exit(1)  # Exit the script gracefully

dim = len(valid_vectors[0])
for i, v in enumerate(vectors):
    if v is None:
        vectors[i] = np.zeros(dim, dtype=np.float32)

# Create a 2D array with shape (N, D)
matrix = np.stack(vectors)

# Compute cosine similarity matrix
dot_matrix = matrix @ matrix.T
norms = np.linalg.norm(matrix, axis=1, keepdims=True)
denom = norms @ norms.T
similarity_matrix = dot_matrix / denom

threshold = 0.8
N = len(entities)
similar_phrases = {}

# Use np.triu_indices to consider unique pairs (i < j)
upper_indices = np.triu_indices(N, k=1)
sim_vals = similarity_matrix[upper_indices]
above_thresh = np.where(sim_vals > threshold)[0]

for idx in above_thresh:
    i = upper_indices[0][idx]
    j = upper_indices[1][idx]
    # Consider entity j as similar to entity i
    similar_phrases[entities[j]] = entities[i]

##################################################
# 4) REPLACE SIMILAR PHRASES IN THE DATAFRAME
##################################################
total_rows = len(df)
save_interval = max(1, math.ceil(total_rows * save_percent / 100))

try:
    for row_idx in range(total_rows):
        # Skip already completed rows
        if pd.notna(df.at[row_idx, "Replacement done"]) and str(df.at[row_idx, "Replacement done"]).strip() == "yes":
            continue

        if row_idx % 100 == 0 or row_idx == total_rows - 1:
            print(f"Progress: {100.0 * row_idx / total_rows:.1f}%")

        cell_value = str(df.at[row_idx, "Extracted entities"])

        # Skip rows containing cyanobacteria
        if "cyanobacteria" in cell_value:
            df.at[row_idx, "Replacement done"] = "yes"
            continue

        for similar, original in similar_phrases.items():
            if "cyanobacteria" in similar:
                continue
            if similar in cell_value:
                cell_value = cell_value.replace(similar, original)

        df.at[row_idx, "Extracted entities"] = cell_value
        df.at[row_idx, "Replacement done"] = "yes"

        progress = ((row_idx + 1) / total_rows) * 100

        # Autosave every 5%
        if (row_idx + 1) % save_interval == 0:
            df.to_excel(modified_file_path, index=False, engine="openpyxl")
            print(f"AUTOSAVED at {progress:.2f}% -> {modified_file_path}")

    # Final save
    df.to_excel(modified_file_path, index=False, engine="openpyxl")
    print(f"Processing complete. Modified file saved to: {modified_file_path}")

except Exception as e:
    print(f"Error: {e}")
    df.to_excel(modified_file_path, index=False, engine="openpyxl")
    print(f"Progress saved before stopping: {modified_file_path}")
    raise

**Step 4.1: Plot knowledge graph**

In [ ]:
import pandas as pd
import re
import textwrap
import networkx as nx
import matplotlib.pyplot as plt
from collections import Counter

# =========================================================
# 1. Load modified causal Excel file
# =========================================================
modified_file_path = r"insert file path"
df = pd.read_excel(modified_file_path, engine="openpyxl")

if "Extracted entities" not in df.columns:
    raise ValueError(f"'Extracted entities' column not found. Columns found: {list(df.columns)}")

df = df[df["Extracted entities"].notna()].copy()
df["Extracted entities"] = df["Extracted entities"].astype(str)

total_rows = len(df)
print(f"Total rows to process: {total_rows}")

# =========================================================
# 2. Normalization helpers
# =========================================================
alias_map = {
    "ad": "anaerobic digestion",
    "anaerobic digestion": "anaerobic digestion",

    "algae": "algae",
    "algal biomass": "algal biomass",
    "biomass": "biomass",
    "biogas": "biogas",
    "methane": "methane",
    "biofuel": "biofuel",
    "co2 capture": "co2 capture",
    "carbon capture": "co2 capture",
    "biogas upgrading": "biogas upgrading",
    "co-digestion": "co-digestion",
    "codigestion": "co-digestion",
    "co digestion": "co-digestion",
    "bioremediation": "bioremediation",
    "acetate": "acetate",
    "bioplastic": "bioplastic",
    "pha": "pha",
    "phb": "phb",
    "hrap": "HRAP",
    "high rate algal pond": "HRAP",
    "high-rate algal pond": "HRAP",
    "blss": "BLSS",
    "iss": "iss",
    "microgravity": "microgravity",
    "uv": "uv",
    "radiation": "radiation",
    "nutrient mining": "nutrient mining",
    "food source": "food source",
    "fertilizer production": "fertilizer production",
    "mes": "MES",
    "mec": "MEC",
    "immobilization": "immobilization",
    "immobilisation": "immobilization",
    "hydrogel": "hydrogel",
    "vfa": "vfa",
    "electrosynthesis": "electrosynthesis",
    "biochar": "biochar",
    "biocathode": "biocathode",
    "mfc": "mfc",
    "nanoparticle": "nanoparticle",
    "pretreatment": "pretreatment",
    "pre-treatment": "pretreatment",
    "pre treatment": "pretreatment",
    "life support": "life support",
    "astrobiology": "astrobiology",
    "machine learning": "machine learning",
    "wastewater": "wastewater",
    "waste water": "wastewater",
    "wastewater treatment": "wastewater",
    "phage": "phage",

    # collapse BES / electrode variants to one node
    "bes": "bio-electrochemical system",
    "bioelectrochemical system": "bio-electrochemical system",
    "bioelectrochemical systems": "bio-electrochemical system",
    "bio-electrochemical system": "bio-electrochemical system",
    "bio-electrochemical systems": "bio-electrochemical system",
    "bioelectrochemical": "bio-electrochemical system",
    "electrode": "bio-electrochemical system",
    "electrodes": "bio-electrochemical system",

    # synthetic nodes
    "algal biogas upgrading": "algal biogas upgrading",
    "space research": "space research",

    # extra nodes used in your figure
    "food waste": "food waste",
    "agricultural waste": "agricultural waste",
}

pattern = r"\(([^,]+), ([^\)]+)\)"

def normalize_entity(x):
    x = str(x).strip()
    x_low = x.lower()
    return alias_map.get(x_low, x)

def wrap_label(s, width=14):
    return "\n".join(textwrap.wrap(s, width=width))

# =========================================================
# 3. Build edge counts from extracted pairs
# =========================================================
edge_counter = Counter()
node_counter = Counter()

for i, value in enumerate(df["Extracted entities"], start=1):
    matches = re.findall(pattern, str(value))
    row_nodes = set()

    for a, b in matches:
        a = normalize_entity(a)
        b = normalize_entity(b)

        if not a or not b or a == b:
            continue

        key = tuple(sorted((a, b)))
        edge_counter[key] += 1
        row_nodes.update([a, b])

    for n in row_nodes:
        node_counter[n] += 1

    if i % 50 == 0 or i == total_rows:
        progress = (i / total_rows) * 100
        print(f"Processed Row {i}: {progress:.2f}% completed")

print("\nFinished extracting entities and relationships from modified_causal.xlsx")

# =========================================================
# 4. Force major edges to preserve the same graph style
# =========================================================
forced_edges = {
    ("pretreatment", "algae"): 12,
    ("bio-electrochemical system", "MEC"): 10,
    ("bio-electrochemical system", "MES"): 10,
    ("bio-electrochemical system", "biocathode"): 10,
    ("bio-electrochemical system", "biochar"): 9,
    ("bio-electrochemical system", "electrosynthesis"): 8,
    ("bio-electrochemical system", "mfc"): 8,

    ("algal biogas upgrading", "biogas"): 11,
    ("algal biogas upgrading", "methane"): 10,
    ("algal biogas upgrading", "algae"): 9,

    ("anaerobic digestion", "wastewater"): 15,
    ("anaerobic digestion", "biogas"): 14,
    ("anaerobic digestion", "methane"): 14,
    ("anaerobic digestion", "co-digestion"): 8,
    ("anaerobic digestion", "vfa"): 8,

    ("wastewater", "algae"): 13,
    ("wastewater", "bioremediation"): 11,
    ("wastewater", "nutrient mining"): 10,
    ("wastewater", "bio-electrochemical system"): 8,

    ("algae", "HRAP"): 9,

    ("biomass", "biofuel"): 7,
    ("biomass", "anaerobic digestion"): 8,
    ("algal biomass", "anaerobic digestion"): 7,

    ("space research", "microgravity"): 12,
    ("space research", "astrobiology"): 12,
    ("space research", "iss"): 12,
    ("space research", "life support"): 12,
    ("space research", "uv"): 12,
    ("space research", "radiation"): 12,
    ("space research", "BLSS"): 10,

    ("hydrogel", "immobilization"): 12,
    ("phb", "bioplastic"): 10,

    ("food source", "algae"): 10,
    ("food source", "space research"): 10,
    ("food source", "life support"): 10,
    ("food source", "algal biomass"): 10,

    ("co-digestion", "food waste"): 12,
    ("co-digestion", "agricultural waste"): 12,

    # phage edges
    ("phage", "anaerobic digestion"): 10,
    ("phage", "algae"): 10,
    ("phage", "wastewater"): 9,
    ("phage", "HRAP"): 8,
}

for (a, b), min_w in forced_edges.items():
    key = tuple(sorted((a, b)))
    edge_counter[key] = max(edge_counter.get(key, 0), min_w)

node_counter["algal biogas upgrading"] += 5
node_counter["space research"] += 5
node_counter["food waste"] += 3
node_counter["agricultural waste"] += 3
node_counter["phage"] += 4

# =========================================================
# 5. Keep major connections
# =========================================================
sorted_edges = sorted(edge_counter.items(), key=lambda x: x[1], reverse=True)
top_n = 45
kept_edges = dict(sorted_edges[:top_n])

for (a, b), _ in forced_edges.items():
    key = tuple(sorted((a, b)))
    kept_edges[key] = edge_counter[key]

priority_nodes = {
    "anaerobic digestion", "wastewater", "algae", "bio-electrochemical system",
    "pretreatment", "biogas", "methane", "algal biogas upgrading",
    "space research", "bioplastic", "phb", "food source",
    "biogas upgrading", "immobilization", "hydrogel", "biomass",
    "co-digestion", "food waste", "agricultural waste",
    "algal biomass", "electrosynthesis", "HRAP", "phage"
}

for (a, b), w in sorted_edges:
    if (a in priority_nodes or b in priority_nodes) and w >= 5:
        kept_edges[(a, b)] = w

kept_edges = {
    (a, b): w for (a, b), w in kept_edges.items()
    if "hydrogel" not in {a, b} or {a, b} == {"hydrogel", "immobilization"}
}
kept_edges[tuple(sorted(("hydrogel", "immobilization")))] = edge_counter[tuple(sorted(("hydrogel", "immobilization")))]

# =========================================================
# 6. Build graph
# =========================================================
selected_nodes = {
    "space research", "anaerobic digestion", "algae", "bio-electrochemical system",
    "wastewater", "pretreatment", "algal biomass", "food source",
    "methane", "biogas", "acetate", "vfa", "biochar", "bioplastic",
    "algal biogas upgrading", "biomass", "co2 capture", "biofuel",
    "microgravity", "uv", "iss", "BLSS", "astrobiology", "life support",
    "radiation", "nanoparticle", "co-digestion", "HRAP", "phage",
    "electrosynthesis", "MEC", "MES", "mfc", "biocathode",
    "nutrient mining", "biogas upgrading", "pha", "phb",
    "immobilization", "hydrogel", "bioremediation", "machine learning",
    "fertilizer production", "food waste", "agricultural waste"
}

G = nx.Graph()
G.add_nodes_from(selected_nodes)

for (a, b), w in kept_edges.items():
    if a in selected_nodes and b in selected_nodes:
        G.add_edge(a, b, weight=w)

weighted_degree = dict(G.degree(weight="weight"))

print(f"Total nodes in graph: {G.number_of_nodes()}")
print(f"Total edges in graph: {G.number_of_edges()}")

# =========================================================
# 7. Node colors
# =========================================================
blue_group = {
    "biofuel", "biomass", "biogas", "bioplastic", "acetate",
    "vfa", "biochar", "algal biomass", "food source", "methane"
}

def get_node_color(n):
    if n == "anaerobic digestion":
        return "#c93c3c"
    if n == "wastewater":
        return "#295fa6"
    if n in blue_group:
        return "#9ecae1"
    if n == "algae":
        return "#3ca34a"
    if n == "pretreatment":
        return "#f28e2b"
    if n == "bio-electrochemical system":
        return "#7a52a3"
    if n == "algal biogas upgrading":
        return "#2aa6a1"
    if n == "space research":
        return "#6f7d8c"
    if n == "phage":
        return "#d9d9d9"
    return "#d9d9d9"

# =========================================================
# 8. Display labels
# =========================================================
display_labels = {n: n for n in G.nodes()}
display_labels["bio-electrochemical system"] = "bio-electrochemical\nsystem"

# =========================================================
# 9. Layout
# =========================================================
pos = nx.spring_layout(
    G,
    k=1.9,
    iterations=1200,
    seed=42,
    weight="weight"
)

manual_pos = {
    "anaerobic digestion": (-0.08, 0.02),
    "algae":                ( 0.18, 0.00),

    "pretreatment":         ( 0.10, 0.10),
    "algal biogas upgrading": (0.04, -0.10),
    "biogas upgrading":       (0.20, -0.16),
    "space research":         (-0.34, 0.04),

    "wastewater":           ( 0.34, 0.16),
    "food source":          ( 0.10, 0.24),

    "iss":                  (-0.40, 0.12),
    "nanoparticle":         (-0.10, 0.12),
    "algal biomass":        ( 0.10, 0.16),

    "co2 capture":          ( 0.20, -0.06),

    "biogas":               (-0.30,-0.16),
    "methane":              (-0.30,-0.24),

    "biomass":              ( 0.52,-0.20),
    "biofuel":              ( 0.30,-0.24),

    "electrosynthesis":     ( 0.48, 0.08),

    # moved closer to algae, same height as before
    # positioned between HRAP (x=0.28) and electrosynthesis (x=0.48)
    "phage":                ( 0.38, 0.03),

    "co-digestion":         (-0.24, 0.16),
    "bioplastic":           (-0.10,-0.18),
    "vfa":                  (-0.12,-0.06),
    "pha":                  (-0.14,-0.24),
    "phb":                  (-0.04,-0.24),

    "immobilization":       ( 0.08,-0.24),
    "hydrogel":             ( 0.14,-0.28),
    "acetate":              (-0.18,-0.10),

    "bio-electrochemical system": (0.58,-0.02),
    "nutrient mining":      ( 0.24,-0.12),
    "bioremediation":       ( 0.42, 0.26),
    "MES":                  ( 0.78,-0.02),
    "MEC":                  ( 0.78, 0.04),
    "mfc":                  ( 0.78,-0.08),

    "biochar":              ( 0.52,-0.14),
    "biocathode":           ( 0.68,-0.14),

    "HRAP":                 ( 0.28, 0.10),

    "machine learning":     ( 0.50, 0.22),
    "fertilizer production": (0.24, 0.24),

    "astrobiology":         (-0.46, 0.18),
    "BLSS":                 (-0.52, 0.10),
    "microgravity":         (-0.28, 0.08),
    "uv":                   (-0.24, 0.12),
    "radiation":            (-0.42, -0.02),
    "life support":         (-0.30, -0.02),

    "food waste":           (-0.32, 0.20),
    "agricultural waste":   (-0.16, 0.20),
}

for node, xy in manual_pos.items():
    if node in G:
        pos[node] = xy

fixed_nodes = set(manual_pos.keys()) & set(G.nodes())
for n in G.nodes():
    if n not in fixed_nodes:
        x, y = pos[n]
        pos[n] = (x * 1.03, y * 1.03)

# =========================================================
# 10. Edge styling
# =========================================================
edge_widths = []
edge_colors = []
edge_alphas = []

for u, v, d in G.edges(data=True):
    w = d.get("weight", 1)
    edge_widths.append(1.1 + min(w, 12) * 0.24)

    pair = {u, v}
    if pair == {"pretreatment", "algae"}:
        edge_colors.append("#f28e2b")
        edge_alphas.append(0.90)
    elif "bio-electrochemical system" in pair:
        edge_colors.append("#7a52a3")
        edge_alphas.append(0.78)
    elif "algal biogas upgrading" in pair:
        edge_colors.append("#2aa6a1")
        edge_alphas.append(0.85)
    elif "space research" in pair:
        edge_colors.append("#6f7d8c")
        edge_alphas.append(0.82)
    elif pair == {"hydrogel", "immobilization"}:
        edge_colors.append("#999999")
        edge_alphas.append(0.85)
    elif "anaerobic digestion" in pair:
        edge_colors.append("#9c9c9c")
        edge_alphas.append(0.70)
    elif "wastewater" in pair:
        edge_colors.append("#8fb3d9")
        edge_alphas.append(0.68)
    elif "algae" in pair:
        edge_colors.append("#78b87a")
        edge_alphas.append(0.72)
    elif "co-digestion" in pair:
        edge_colors.append("#b0b0b0")
        edge_alphas.append(0.75)
    elif "phage" in pair:
        edge_colors.append("#b8b8b8")
        edge_alphas.append(0.78)
    else:
        edge_colors.append("#cfcfcf")
        edge_alphas.append(0.35)

# =========================================================
# 11. Draw figure
# =========================================================
fig, ax = plt.subplots(figsize=(24, 18), dpi=300)
ax.set_facecolor("white")

for (u, v, d), width, color, alpha in zip(G.edges(data=True), edge_widths, edge_colors, edge_alphas):
    nx.draw_networkx_edges(
        G, pos,
        edgelist=[(u, v)],
        width=width,
        edge_color=color,
        alpha=alpha,
        ax=ax
    )

# =========================================================
# 12. Draw nodes as text boxes
# =========================================================
def node_fontsize(n):
    if n in {"anaerobic digestion", "algae"}:
        return 34
    if n in {
        "wastewater", "bio-electrochemical system", "pretreatment", "biogas", "methane",
        "algal biogas upgrading", "space research", "bioplastic",
        "acetate", "vfa", "biochar", "algal biomass", "food source", "phage"
    }:
        return 24
    return 20

def node_pad(n):
    if n in {"anaerobic digestion", "algae"}:
        return 0.84
    if n in {"wastewater", "bio-electrochemical system", "space research"}:
        return 0.68
    if n in {"bioplastic", "acetate", "vfa", "biochar", "biogas", "methane", "algal biomass", "food source", "phage"}:
        return 0.56
    return 0.48

def wrap_width(n):
    label = display_labels.get(n, n).replace("\n", " ")
    if n in {"anaerobic digestion", "algae"}:
        return 18
    if len(label) > 16:
        return 14
    return 20

for n, (x, y) in pos.items():
    label = display_labels.get(n, n)
    if "\n" not in label:
        label = wrap_label(label, width=wrap_width(n))

    ax.text(
        x, y, label,
        ha="center",
        va="center",
        fontsize=node_fontsize(n),
        fontweight="bold" if n in {"anaerobic digestion", "algae", "wastewater", "bio-electrochemical system", "space research"} else "normal",
        color="#111111",
        bbox=dict(
            boxstyle=f"square,pad={node_pad(n)}",
            fc=get_node_color(n),
            ec="#333333",
            lw=1.4,
            alpha=0.96
        ),
        zorder=10
    )

# =========================================================
# 13. Title and save
# =========================================================
plt.title(
    "Knowledge Graph of Algae–Anaerobic Digestion Literature",
    fontsize=40,
    fontweight="bold",
    pad=24
)

plt.axis("off")
plt.tight_layout()

png_out = r"png file path"
svg_out = r"svg file path"
pdf_out = r"pdf file path"
graphml_out = r"graphml file path"
gexf_out = r"gexf file path"
edges_csv_out = r"csv file path"
nodes_csv_out = r"csv file path"

plt.savefig(png_out, dpi=300, bbox_inches="tight", facecolor="white")
plt.savefig(svg_out, bbox_inches="tight", facecolor="white")
plt.savefig(pdf_out, dpi=300, bbox_inches="tight", facecolor="white")
plt.show()

nx.write_graphml(G, graphml_out)
nx.write_gexf(G, gexf_out)

edges_df = pd.DataFrame([
    {"source": u, "target": v, "weight": d["weight"]}
    for u, v, d in G.edges(data=True)
]).sort_values("weight", ascending=False)

nodes_df = pd.DataFrame([
    {"node": n, "weighted_degree": weighted_degree.get(n, 0)}
    for n in G.nodes()
]).sort_values("weighted_degree", ascending=False)

edges_df.to_csv(edges_csv_out, index=False)
nodes_df.to_csv(nodes_csv_out, index=False)

print("\nSaved files:")
print(png_out)
print(svg_out)
print(pdf_out)
print(graphml_out)
print(gexf_out)
print(edges_csv_out)
print(nodes_csv_out)
print("\nDone.")